In [ ]:
!pip install -q transformers datasets accelerate evaluate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
# 1.Load and inspect the dataset
# Load the emotion classification dataset from Hugging Face
dataset = load_dataset("emotion")

# Print dataset sizes to understand the data split
print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"Validation samples: {len(dataset['validation'])}")

# Inspect a single training sample
print("\nSample:")
print(dataset['train'][0])

# Extract label names for later use
label_names = dataset['train'].features['label'].names
print(f"\nLabels: {label_names}")

In [ ]:
# 2.Model and tokenizer selection
# Use a lightweight BERT based model suitable for text classification
model_name = "distilbert-base-uncased"
num_labels = len(dataset['train'].features['label'].names)


# Load tokenizer and model with correct number of output labels
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

In [ ]:
# 3.Text tokenization
# Tokenization function converts raw text into model input format
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Apply tokenization to the entire dataset
# Remove raw text column since the model only needs tokenized inputs
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=['text'])

In [ ]:
# 4.Training configuration
# Define training arguments for fine tuning
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,  # Reduced batch size for better learning
    per_device_eval_batch_size=16,
    num_train_epochs=8,  # Increased from 3 to 8
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    warmup_steps=500,  # Added warmup
    save_total_limit=2,  # Save only best 2 models
)

# 5.Evaluation metrics
# Custom metric function for accuracy and F1 score
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted')
    }

In [ ]:
# 6.Trainer initialization
# Hugging Face Trainer handles training and evaluation loop
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics,
)

# 7.Model training
trainer.train()

In [ ]:
# 8.Final evaluation on test set
# Fixed classification report
results = trainer.evaluate(tokenized_datasets['test'])
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1 Score: {results['eval_f1']:.4f}")

# Generate predictions
predictions = trainer.predict(tokenized_datasets['test'])
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = tokenized_datasets['test']['label']

# Check unique labels in test set
unique_labels = sorted(set(true_labels))
print(f"\nUnique labels in test set: {unique_labels}")
print(f"Number of unique labels: {len(unique_labels)}")

# Create label names for only the classes present in test set
present_label_names = [label_names[i] for i in unique_labels]

print("\n" + classification_report(
    true_labels,
    pred_labels,
    labels=unique_labels,
    target_names=present_label_names
))

In [ ]:
# 9.Inference function
# Function for predicting emotion from raw text
def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # Move inputs to GPU if available
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    # Disable gradient calculation for inference
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract predicted class and confidence
    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    confidence = torch.softmax(outputs.logits, dim=1)[0][predicted_class].item()
    return label_names[predicted_class], confidence

In [ ]:
# Test the chatbot with clearer emotions
test_cases = [
    "I am so angry right now!",
    "This is absolutely disgusting!",
    "I am terrified and very scared!",
    "I am so happy and excited today!",
    "I feel so sad and depressed.",
    "Wow, that was completely unexpected!",
]

for text in test_cases:
    emotion, confidence = predict_emotion(text)
    print(f"Text: {text}")
    print(f"→ {emotion.upper()} ({confidence:.1%})\n")

In [ ]:
# Save model and tokenizer for later use or deployment
trainer.save_model("./trained_model")
tokenizer.save_pretrained("./trained_model")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Trained this model to classify emotions from text
MODEL_PATH = "/content/a/trained_model"

print("Loading model...")

# Tokenizer converts text into input IDs that the model understands
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, local_files_only=True)


label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]

# Move model to the chosen device and set it to evaluation mode
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval() # evaluation mode disables dropout
print(f"Model loaded! Using: {device}")

In [ ]:
def predict_emotion(text):
    if not text.strip():
        return None, 0.0

    # Tokenize the input text for the model
    # truncation/padding ensures consistent input size
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Run the model without computing gradients (inference mode)
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert logits to probabilities
    probabilities = torch.softmax(outputs.logits, dim=1)[0]
    # Get the predicted class index
    predicted_class = torch.argmax(probabilities).item()
    # Get confidence of the prediction
    confidence = probabilities[predicted_class].item()

    return label_names[predicted_class], confidence

In [ ]:
# These make the chatbot feel human-like
responses = {
    "sadness": "I'm sorry you're feeling down. I hope things get better soon.😔",
    "joy": "That sounds wonderful! I'm glad you're feeling happy!😊",
    "love": "That's so sweet! Love is a beautiful thing!❤️",
    "anger": "I can understand your frustration. Take a deep breath.😤",
    "fear": "That sounds concerning. I hope everything works out for you.🤗",
    "surprise": "Wow, that's quite unexpected!😲"
}

In [ ]:
# Display a welcome message and instructions to the user
print("-" * 60)
print("EMOTION DETECTION CHATBOT - NLU PROJECT")
print("-" * 60)
print("\nType your message to analyze emotions!")
print("Type 'exit' to stop\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("\nBot: Goodbye! 👋")
        break

    if not user_input:
        continue

    emotion, confidence = predict_emotion(user_input)
    bot_response = responses.get(emotion, "I understand.")

    print(f"Emotion: {emotion.upper()} | Confidence: {confidence:.1%}")
    print(f"Bot: {bot_response}\n")